# Qwen3-VL Answers

Choose an experiment. Which claims are directly supported by the image?

Run the setup cells, then change the input in **Demo code**. All packages, models and data stay in the Colab runtime. For large models, select a GPU runtime. Free GPU availability varies.

Source: https://huggingface.co/Qwen/Qwen3-VL-2B-Instruct

## Runtime setup

In [ ]:
# Packages are installed in the Colab runtime, not on your computer.
%pip -q install "matplotlib>=3.9,<3.12" "scikit-learn==1.6.1" "Pillow" "requests" "tqdm" "transformers==4.57.1" "huggingface-hub==0.36.2" "accelerate" "safetensors" "qwen-vl-utils"
import os
os.environ["TABPFN_DISABLE_TELEMETRY"] = "1"
os.environ["WANDB_MODE"] = "disabled"
import torch
print("Runtime:", "GPU " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
import os
import random
import tarfile
import requests
import json
import time
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn.functional as F
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,   # LLM
    Qwen3VLForConditionalGeneration,        # Multimodal LLM
    AutoProcessor,
)
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name()}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## Input images

Fixed Oxford-IIIT Pet classroom subset. Source: https://www.robots.ox.ac.uk/~vgg/data/pets/

In [ ]:
from pathlib import Path
import urllib.request, zipfile

IMG_DIR = "./data"
NUM_IMAGES_PER_CLASS = 50
batch_size = 16
if not all(len(list((Path(IMG_DIR) / label).glob("*.jpg"))) == 50 for label in ["dog", "cat"]):
    url = "https://moai.postech.ac.kr/teaching/27s-programming-for-ax/labs/colab/oxford-pets-classroom.zip"
    archive, _ = urllib.request.urlretrieve(url)
    with zipfile.ZipFile(archive) as bundle:
        bundle.extractall(".")
image_paths, image_labels = [], []
for label in ["dog", "cat"]:
    paths = sorted((Path(IMG_DIR) / label).glob("*.jpg"))
    image_paths.extend(str(path) for path in paths)
    image_labels.extend([label] * len(paths))
print(f"Classroom subset: {len(image_paths)} images")


In [ ]:
VLM_ID = "Qwen/Qwen3-VL-2B-Instruct"

print(f"📦 Loading Qwen3-VL: {VLM_ID}")
print(f"   (the model download takes a few minutes on first run)\n")

vlm_processor = AutoProcessor.from_pretrained(VLM_ID, use_fast=False, min_pixels=64 * 32 * 32, max_pixels=256 * 32 * 32)

vlm_model = Qwen3VLForConditionalGeneration.from_pretrained(
    VLM_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)

if not torch.cuda.is_available():
    vlm_model = vlm_model.to(device)

vlm_params = sum(p.numel() for p in vlm_model.parameters())
if device.type == 'cpu':
    torch.backends.quantized.engine = 'fbgemm'
    vlm_model = torch.ao.quantization.quantize_dynamic(vlm_model, {torch.nn.Linear}, dtype=torch.qint8, inplace=True)
    print("CPU linear weights: dynamic int8; embeddings and convolutions: float32")
vlm_model.eval()
print(f"   ✅ Loaded!")
print(f"   - Parameters: {vlm_params / 1e9:.1f}B")
print(f"   - dtype: {next(vlm_model.parameters()).dtype}")

In [ ]:
from qwen_vl_utils import process_vision_info

def ask_vlm(image_path, question, max_new_tokens=48, temperature=0.7):
    """
    Send an image and a question to Qwen3-VL and get the response.
    
    Args:
        image_path: str, path to the image file
        question: str, the question text
    Returns:
        str: model response
    """
    if "JSON" not in question:
        question += "\nAnswer concisely in at most 25 words."
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": f"file://{os.path.abspath(image_path)}"},
                {"type": "text", "text": question},
            ],
        }
    ]
    
    text = vlm_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    
    inputs = vlm_processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(vlm_model.device)
    
    with torch.no_grad():
        generated_ids = vlm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True if temperature > 0 else False,
        )
    
    # Strip the input portion
    generated_ids_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    
    response = vlm_processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]
    
    return response.strip()

print("✅ ask_vlm function defined")

## Demo code

Choose an experiment. Which claims are directly supported by the image?

In [ ]:
question = "What animal is visible? Describe the background."
answer = ask_vlm(image_paths[0], question, max_new_tokens=48)
print(answer)

In [ ]:
# Generate captions for Dog/Cat images
print("📸 Qwen3-VL Image Captioning\n")

caption_samples = [0, 2, NUM_IMAGES_PER_CLASS, NUM_IMAGES_PER_CLASS + 2]

fig, axes = plt.subplots(1, len(caption_samples), figsize=(5 * len(caption_samples), 6))
fig.suptitle('📸 Qwen3-VL — Image Captioning', fontsize=16, fontweight='bold')

for i, idx in enumerate(caption_samples):
    img = Image.open(image_paths[idx]).convert('RGB')
    gt = image_labels[idx].upper()
    
    caption = ask_vlm(image_paths[idx], "Describe this image in one detailed sentence.")
    
    print(f"Image: {image_paths[idx]} | GT: {gt}\nCaption: {caption}\n")
    axes[i].imshow(img)
    # Wrap long captions
    wrapped = '\n'.join([caption[j:j+35] for j in range(0, min(len(caption), 140), 35)])
    axes[i].set_title(f'[{gt}]\n{wrapped}', fontsize=9, fontweight='bold')
    axes[i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Various questions about a single image
sample_path = image_paths[0]
sample_img = Image.open(sample_path).convert('RGB')
sample_gt = image_labels[0]

questions = [
    "What animal is in this image?",
    "What breed do you think this is?",
    "Describe the background of this image.",
    "What emotion does this animal seem to be expressing?",
    "Is this animal indoors or outdoors?",
    "Estimate the age of this animal (puppy/adult/senior).",
]

print(f"🐾 VQA experiment — {sample_gt.upper()} image\n")

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(sample_img)
ax.set_title(f'Test Image ({sample_gt.upper()})', fontsize=12, fontweight='bold')
ax.axis('off')
plt.show()

print("\n❓ Questions & answers:\n")
for q in questions:
    answer = ask_vlm(sample_path, q, max_new_tokens=48)
    print(f"  Q: {q}")
    print(f"  A: {answer}")
    print()

In [ ]:
def ask_vlm_multi_image(image_paths_list, question, max_new_tokens=128):
    """Show multiple images at once and ask a question."""
    content = []
    for i, path in enumerate(image_paths_list):
        content.append({"type": "image", "image": f"file://{os.path.abspath(path)}"})
        content.append({"type": "text", "text": f"(Image {i+1})"})
    content.append({"type": "text", "text": question})
    
    messages = [{"role": "user", "content": content}]
    
    text = vlm_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = vlm_processor(
        text=[text], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt"
    ).to(vlm_model.device)
    
    with torch.no_grad():
        gen_ids = vlm_model.generate(**inputs, max_new_tokens=max_new_tokens)
    
    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, gen_ids)]
    return vlm_processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()

# Compare a Dog vs Cat image
dog_path = image_paths[0]
cat_path = image_paths[NUM_IMAGES_PER_CLASS]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(Image.open(dog_path)); axes[0].set_title('Image 1 (Dog)', fontweight='bold'); axes[0].axis('off')
axes[1].imshow(Image.open(cat_path)); axes[1].set_title('Image 2 (Cat)', fontweight='bold'); axes[1].axis('off')
plt.suptitle('🔍 Image comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

comparison = ask_vlm_multi_image(
    [dog_path, cat_path],
    "Compare these two animals. What are the differences and similarities between Image 1 and Image 2?"
)
print(f"\n🔍 Comparison result:\n{comparison}")

In [ ]:
json_prompt = """Analyze this image and respond ONLY with a JSON object (no markdown, no explanation) with these fields:
- "animal": the type of animal ("dog" or "cat")
- "breed_guess": your best guess of the breed
- "color": main color of the animal
- "setting": "indoor" or "outdoor"
- "mood": the apparent mood of the animal
- "confidence": your confidence in the animal type (0-100)"""

print("📋 Generating structured output (JSON)\n")

json_samples = [0, NUM_IMAGES_PER_CLASS]  # one dog and one cat

fig, axes = plt.subplots(1, len(json_samples), figsize=(5 * len(json_samples), 5))

for i, idx in enumerate(json_samples):
    img = Image.open(image_paths[idx]).convert('RGB')
    gt = image_labels[idx].upper()
    
    response = ask_vlm(image_paths[idx], json_prompt, max_new_tokens=160, temperature=0.1)
    
    axes[i].imshow(img)
    axes[i].set_title(f'GT: {gt}', fontsize=11, fontweight='bold')
    axes[i].axis('off')
    
    print(f"  [{gt}] {os.path.basename(image_paths[idx])}:")
    
    # Try to parse JSON
    try:
        # Extract only the JSON part
        json_str = response
        if '{' in json_str:
            json_str = json_str[json_str.index('{'):json_str.rindex('}')+1]
        parsed = json.loads(json_str)
        print(f"  {json.dumps(parsed, indent=4)}")
    except (json.JSONDecodeError, ValueError):
        print(f"  (Raw) {response[:200]}")
    print()

plt.tight_layout()
plt.show()

In [ ]:
def multi_turn_conversation(image_path, conversation_questions):
    """
    Run a multi-turn conversation about an image.
    """
    messages = []
    results = []
    
    for i, question in enumerate(conversation_questions):
        if i == 0:
            # Include the image only in the first question
            messages.append({
                "role": "user",
                "content": [
                    {"type": "image", "image": f"file://{os.path.abspath(image_path)}"},
                    {"type": "text", "text": question},
                ],
            })
        else:
            messages.append({
                "role": "user",
                "content": [{"type": "text", "text": question}],
            })
        
        text = vlm_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = vlm_processor(
            text=[text], images=image_inputs, videos=video_inputs,
            padding=True, return_tensors="pt"
        ).to(vlm_model.device)
        
        with torch.no_grad():
            gen_ids = vlm_model.generate(**inputs, max_new_tokens=96)
        
        trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, gen_ids)]
        response = vlm_processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()
        
        # Add the response to the conversation history
        messages.append({"role": "assistant", "content": [{"type": "text", "text": response}]})
        results.append((question, response))
    
    return results

# Run a multi-turn conversation
sample_path = image_paths[0]

conversation = [
    "What do you see in this image?",
    "What breed do you think it is and why?",
    "What kind of food would be best for this animal?",
    "Write a short funny caption for this image.",
]

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(Image.open(sample_path))
ax.set_title('Conversation Subject', fontweight='bold')
ax.axis('off')
plt.show()

print("\n💬 Multi-turn conversation:\n")
results = multi_turn_conversation(sample_path, conversation)
for i, (q, a) in enumerate(results):
    print(f"  👤 Turn {i+1}: {q}")
    print(f"  🤖 {a}")
    print()

In [ ]:
# Run a VLM classification experiment on a subset of images
n_test = min(8, len(image_paths))  # VLM is slow, so only a subset
test_indices = random.sample(range(len(image_paths)), n_test)

classify_prompt = 'Is this a dog or a cat? Answer with only one word: "dog" or "cat".'

print(f"🏷️ VLM Classification ({n_test} images)\n")

vlm_preds = []
vlm_gts = []

for idx in tqdm(test_indices, desc='Classifying'):
    response = ask_vlm(image_paths[idx], classify_prompt, max_new_tokens=10, temperature=0.1)
    clean = response.strip().lower().rstrip('.')
    pred = clean if clean in {'dog', 'cat'} else 'unknown'
    vlm_preds.append(pred)
    vlm_gts.append(image_labels[idx])

# Compute accuracy
correct = sum(1 for p, g in zip(vlm_preds, vlm_gts) if p == g)
total = len(vlm_gts)
acc = correct / total

print(f"\n{'='*50}")
print(f"  🏷️ VLM Classification Accuracy: {acc:.1%} ({correct}/{total})")
print(f"{'='*50}")

# Visualize: a subset of the results
n_show = min(8, n_test)
fig, axes = plt.subplots(2, n_show // 2, figsize=(4 * (n_show // 2), 8))
fig.suptitle(f'🏷️ Qwen3-VL Classification (Acc: {acc:.1%})', fontsize=14, fontweight='bold')

for i in range(n_show):
    ax = axes[i // (n_show // 2), i % (n_show // 2)]
    idx = test_indices[i]
    img = Image.open(image_paths[idx]).convert('RGB')
    ax.imshow(img)
    
    gt, pred = vlm_gts[i], vlm_preds[i]
    ok = gt == pred
    ax.set_title(f"{'✓' if ok else '✗'} GT:{gt.upper()} Pred:{pred.upper()}",
                color='green' if ok else 'red', fontsize=10, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()

## Interpretation

Recorded model responses are evidence to inspect, not verified facts.

Compare the output with the evidence in the input. A generated answer can be wrong.